In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str((Path.cwd() / "..").resolve()))

from _infra.nbtools import run, mlir_opt_path, tools, artifacts_dir

ART = artifacts_dir()
SRC = Path("../assets/ir/vecadd_linalg.mlir").resolve()

def save_step(name: str, txt: str) -> Path:
    p = ART / f"{name}.mlir"
    p.write_text(txt)
    print("Wrote:", p)
    return p

In [2]:
from _infra.nbtools import run, mlir_opt_path

pipeline = [
    "-linalg-generalize-named-ops",
    "-one-shot-bufferize=bufferize-function-boundaries",
    "-bufferization-lower-deallocations",
    "-canonicalize", "-cse",
]

txt_buf = run([mlir_opt_path(), *pipeline, str(SRC)])
p2 = save_step("02_bufferized", txt_buf)
print("\nUsed pipeline:\n", " ".join(pipeline))
print("\nPreview:\n", txt_buf[:800])

Wrote: /home/superuser/Code/Cinnamon/tutorial/artifacts/02_bufferized.mlir

Used pipeline:
 -linalg-generalize-named-ops -one-shot-bufferize=bufferize-function-boundaries -bufferization-lower-deallocations -canonicalize -cse

Preview:
 #map = affine_map<(d0) -> (d0)>
module {
  func.func @vecadd(%arg0: memref<104857600xf32, strided<[?], offset: ?>>, %arg1: memref<104857600xf32, strided<[?], offset: ?>>) -> memref<104857600xf32> {
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<104857600xf32>
    linalg.generic {indexing_maps = [#map, #map, #map], iterator_types = ["parallel"]} ins(%arg0, %arg1 : memref<104857600xf32, strided<[?], offset: ?>>, memref<104857600xf32, strided<[?], offset: ?>>) outs(%alloc : memref<104857600xf32>) {
    ^bb0(%in: f32, %in_0: f32, %out: f32):
      %0 = arith.addf %in, %in_0 : f32
      linalg.yield %0 : f32
    }
    return %alloc : memref<104857600xf32>
  }
}




In [3]:
pipeline = [
    "-convert-linalg-to-affine-loops",
    "-canonicalize", "-cse",
]
txt_affine = run([mlir_opt_path(), *pipeline, str(p2)])
p3 = save_step("03_affine", txt_affine)
print(txt_affine[:800])

Wrote: /home/superuser/Code/Cinnamon/tutorial/artifacts/03_affine.mlir
module {
  func.func @vecadd(%arg0: memref<104857600xf32, strided<[?], offset: ?>>, %arg1: memref<104857600xf32, strided<[?], offset: ?>>) -> memref<104857600xf32> {
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<104857600xf32>
    affine.for %arg2 = 0 to 104857600 {
      %0 = affine.load %arg0[%arg2] : memref<104857600xf32, strided<[?], offset: ?>>
      %1 = affine.load %arg1[%arg2] : memref<104857600xf32, strided<[?], offset: ?>>
      %2 = arith.addf %0, %1 : f32
      affine.store %2, %alloc[%arg2] : memref<104857600xf32>
    }
    return %alloc : memref<104857600xf32>
  }
}




In [4]:
pipeline = [
    "-lower-affine",
    "-canonicalize", "-cse",
]
txt_scf = run([mlir_opt_path(), *pipeline, str(p3)])
p4 = save_step("04_scf", txt_scf)
print(txt_scf[:800])

Wrote: /home/superuser/Code/Cinnamon/tutorial/artifacts/04_scf.mlir
module {
  func.func @vecadd(%arg0: memref<104857600xf32, strided<[?], offset: ?>>, %arg1: memref<104857600xf32, strided<[?], offset: ?>>) -> memref<104857600xf32> {
    %c1 = arith.constant 1 : index
    %c104857600 = arith.constant 104857600 : index
    %c0 = arith.constant 0 : index
    %alloc = memref.alloc() {alignment = 64 : i64} : memref<104857600xf32>
    scf.for %arg2 = %c0 to %c104857600 step %c1 {
      %0 = memref.load %arg0[%arg2] : memref<104857600xf32, strided<[?], offset: ?>>
      %1 = memref.load %arg1[%arg2] : memref<104857600xf32, strided<[?], offset: ?>>
      %2 = arith.addf %0, %1 : f32
      memref.store %2, %alloc[%arg2] : memref<104857600xf32>
    }
    return %alloc : memref<104857600xf32>
  }
}




In [5]:
from _infra.nbtools import run, mlir_opt_path

pipeline = [
    "-convert-scf-to-cf",
    "-llvm-request-c-wrappers",
    "-convert-to-llvm",
    "-reconcile-unrealized-casts",
    "-canonicalize",
]

txt_llvm_dialect = run([mlir_opt_path(), *pipeline, str(p4)])
p5 = save_step("05_llvm_dialect", txt_llvm_dialect)
print("\nUsed pipeline:\n", " ".join(pipeline))
print("\nPreview:\n", txt_llvm_dialect[:800])

Wrote: /home/superuser/Code/Cinnamon/tutorial/artifacts/05_llvm_dialect.mlir

Used pipeline:
 -convert-scf-to-cf -llvm-request-c-wrappers -convert-to-llvm -reconcile-unrealized-casts -canonicalize

Preview:
 module {
  llvm.func @malloc(i64) -> !llvm.ptr
  llvm.func @vecadd(%arg0: !llvm.ptr, %arg1: !llvm.ptr, %arg2: i64, %arg3: i64, %arg4: i64, %arg5: !llvm.ptr, %arg6: !llvm.ptr, %arg7: i64, %arg8: i64, %arg9: i64) -> !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)> attributes {llvm.emit_c_interface} {
    %0 = llvm.mlir.constant(64 : index) : i64
    %1 = llvm.mlir.zero : !llvm.ptr
    %2 = llvm.mlir.constant(0 : index) : i64
    %3 = llvm.mlir.constant(104857600 : index) : i64
    %4 = llvm.mlir.constant(1 : index) : i64
    %5 = llvm.mlir.poison : !llvm.struct<(ptr, ptr, i64, array<1 x i64>, array<1 x i64>)>
    %6 = llvm.getelementptr %1[104857600] : (!llvm.ptr) -> !llvm.ptr, f32
    %7 = llvm.ptrtoint %6 : !llvm.ptr to i64
    %8 = llvm.add %7, %0 : i64
    %9 = llvm.

In [6]:
mlir_translate = tools().get("mlir-translate")

ll_path = ART / "06_llvm_ir.ll"
ll_txt = run([mlir_translate, "--mlir-to-llvmir", str(p5)])

if isinstance(ll_txt, (bytes, bytearray)):
    ll_txt = ll_txt.decode("utf-8", errors="replace")

ll_path.write_text(ll_txt)
print("Wrote:", ll_path)
print("\n===== LLVM IR (full) =====\n")
print(ll_txt if ll_txt else "(empty)")


Wrote: /home/superuser/Code/Cinnamon/tutorial/artifacts/06_llvm_ir.ll

===== LLVM IR (full) =====

; ModuleID = 'LLVMDialectModule'
source_filename = "LLVMDialectModule"

declare ptr @malloc(i64)

define { ptr, ptr, i64, [1 x i64], [1 x i64] } @vecadd(ptr %0, ptr %1, i64 %2, i64 %3, i64 %4, ptr %5, ptr %6, i64 %7, i64 %8, i64 %9) {
  %11 = call ptr @malloc(i64 419430464)
  %12 = ptrtoint ptr %11 to i64
  %13 = add i64 %12, 63
  %14 = urem i64 %13, 64
  %15 = sub i64 %13, %14
  %16 = inttoptr i64 %15 to ptr
  %17 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } poison, ptr %11, 0
  %18 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %17, ptr %16, 1
  %19 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %18, i64 0, 2
  %20 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %19, i64 104857600, 3, 0
  %21 = insertvalue { ptr, ptr, i64, [1 x i64], [1 x i64] } %20, i64 1, 4, 0
  br label %22

22:                                               ; preds = %25, %10
  %23 = phi 